# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [16]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [17]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [18]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [19]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- word_count <= 669.50
|   |   |   |--- class: 0
|   |   |--- word_count >  669.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- avg_position <= 38.45
|   |   |   |--- class: 0
|   |   |--- avg_position >  38.45
|   |   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [20]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.850
Precision@50:  hand rule 0.680   vs   tree 0.740


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [21]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

# Your experiment here
hey !! i hope you are doing well.
1) yes,we can read  the tree at 3 and it actually improve precision@50 from 60.00 to 72.00 and at 4 we still read the tree
 but it get so bigger and it improve to 68.00.

2) after changing impressions_90d with engagement_rate precision_at_k is improved in decission tree which suggests that
 engagement_rate is more important than impressions_90d.




### Client-Holdout Validation (Task 3)

Instead of a simple random split, we'll perform a client-holdout split. This ensures that the model is evaluated on entirely unseen clients, which is a more realistic scenario for generalization.


In [23]:
from sklearn.model_selection import train_test_split

# 1. Get unique client IDs
unique_clients = df['client_id'].unique()

# 2. Split client IDs into train and test sets
# We use random_state for reproducibility
train_clients, test_clients = train_test_split(
    unique_clients, test_size=0.2, random_state=42
)

print(f"Total unique clients: {len(unique_clients)}")
print(f"Clients in training set: {len(train_clients)}")
print(f"Clients in test set: {len(test_clients)}")

# 3. Create training and testing DataFrames based on the client splits
train_df = df[df['client_id'].isin(train_clients)].copy()
test_df = df[df['client_id'].isin(test_clients)].copy()

print(f"\nTraining DataFrame shape: {train_df.shape}")
print(f"Testing DataFrame shape: {test_df.shape}")

# Display the first few rows of the training DataFrame
print("\nTraining DataFrame head:")
display(train_df.head())

# Display the first few rows of the testing DataFrame
print("\nTesting DataFrame head:")
display(test_df.head())


Total unique clients: 32
Clients in training set: 25
Clients in test set: 7

Training DataFrame shape: (26581, 46)
Testing DataFrame shape: (3419, 46)

Training DataFrame head:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,hand_rule_score
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,0
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,0
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,0
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,0



Testing DataFrame head:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,hand_rule_score
13,content_a5a2fbc76336,client_8527a891e2,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,...,39.8,0.00,25.00,0.0,moderate,page_3_5,stable,10.4,0,0
15,content_689414059706,client_9f14025af0,0.0,0.00,LOW,0.00,keyword article,informational,2661.0,17889.0,...,7.8,33.33,26.67,0.0,low,page_1,up,675.0,0,0
32,content_5eeba5d398f2,client_bbb965ab0c,0.0,0.00,LOW,0.00,keyword article,informational,2949.0,21532.0,...,28.3,0.00,0.00,0.0,moderate,page_3_5,down,-40.0,1,0
39,content_4595e8704e07,client_8527a891e2,90.0,0.06,LOW,0.03,keyword article,informational,3666.0,21824.0,...,36.3,0.00,0.00,0.0,low,page_3_5,down,-100.0,1,0
40,content_4df0b7207fe3,client_9f14025af0,10.0,0.00,LOW,0.00,keyword article,transactional,896.0,6300.0,...,49.6,16.13,18.18,0.0,moderate,page_3_5,up,197.7,0,0


Now that you have `train_df` and `test_df`, you would extract your features (`X_train`, `X_test`) and labels (`y_train`, `y_test`) from these respective DataFrames. For example:


In [24]:
# Define features (same as before)
features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]

# Prepare training data
X_train = train_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df["is_declining_label"].values

# Prepare testing data
X_test = test_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["is_declining_label"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


X_train shape: (26581, 6)
y_train shape: (26581,)
X_test shape: (3419, 6)
y_test shape: (3419,)


With `X_train`, `y_train`, `X_test`, and `y_test` prepared, you can now train your decision tree model on the training data (`X_train`, `y_train`) and evaluate its `precision_at_k` using the test data (`X_test`, `y_test`). This will give you a more robust understanding of how your model generalizes to unseen clients.

### Train and Evaluate Decision Tree with Client-Holdout Validation

Now we'll re-train the decision tree using the `X_train` and `y_train` from our client-holdout split, and then evaluate its performance on `X_test` and `y_test`.

In [25]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Re-initialize and train the DecisionTreeClassifier on the training data
tree_holdout = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_holdout.fit(X_train, y_train)

# Print the tree learned from the training data
print("\nDecision Tree (trained on client-holdout data):\n")
print(export_text(tree_holdout, feature_names=features))

# Predict probabilities on the test set
tree_holdout_score = tree_holdout.predict_proba(X_test)[:, 1]

# Calculate precision@k for the hand rule (on test_df) and the new tree model
# Need to re-calculate hand_rule_score for the test_df
# Note: y_test is already the labels for the test set

stale_test   = (test_df["days_since_last_update"] >= 180).astype(int)
visible_test = (test_df["impressions_90d"] >= 500).astype(int)
test_df["hand_rule_score_test"] = stale_test * visible_test * test_df["impressions_90d"]

for k in (20, 50):
    hr_test = precision_at_k(test_df['hand_rule_score_test'], y_test, k)
    tr_holdout = precision_at_k(tree_holdout_score, y_test, k)
    print(f"\nPrecision@{k} (Client-Holdout Test Set):  Hand rule {hr_test:.3f}   vs   Tree {tr_holdout:.3f}")



Decision Tree (trained on client-holdout data):

|--- avg_position <= 0.55
|   |--- word_count <= 687.00
|   |   |--- ctr <= 0.04
|   |   |   |--- class: 0
|   |   |--- ctr >  0.04
|   |   |   |--- class: 1
|   |--- word_count >  687.00
|   |   |--- avg_position <= 0.25
|   |   |   |--- class: 0
|   |   |--- avg_position >  0.25
|   |   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- ctr <= 0.31
|   |   |   |--- class: 1
|   |   |--- ctr >  0.31
|   |   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- days_since_last_update <= 25.50
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  25.50
|   |   |   |--- class: 1


Precision@20 (Client-Holdout Test Set):  Hand rule 0.650   vs   Tree 0.550

Precision@50 (Client-Holdout Test Set):  Hand rule 0.620   vs   Tree 0.520


This comparison using the client-holdout validation provides a much more robust evaluation of your model's performance on unseen data and clients, reflecting its true generalization ability.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.